# 02 - Train YOLOv8 fracture detector

Run this notebook on a GPU runtime (Colab or Kaggle). Expects the dataset downloaded by `01_dataset_setup.ipynb` to already be in place.

Outputs:
- Training runs saved under `runs/fracture/train/`
- Best weights copied to `weights/fracture_yolov8n_best.pt`
- On Colab both folders live in Google Drive and persist after the runtime stops.

In [ ]:
RUN_ENV = "local"  # "colab" or "kaggle" recommended for GPU training
PROJECT_NAME = "yolov8-fracture-detection"

MODEL_NAME = "yolov8n.pt"
EPOCHS = 30
IMAGE_SIZE = 640
BATCH_SIZE = 16


In [ ]:
from pathlib import Path
import shutil

if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
elif RUN_ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working") / PROJECT_NAME
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DATA_ROOT = PROJECT_ROOT / "data"
RUNS_ROOT = PROJECT_ROOT / "runs"
WEIGHTS_ROOT = PROJECT_ROOT / "weights"
WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)

# Locate the YAML from the Roboflow download (01_dataset_setup.ipynb)
fracture_dir = DATA_ROOT / "fracture"
yaml_files = list(fracture_dir.glob("*.yaml"))
if not yaml_files:
    raise FileNotFoundError(f"No YAML found in {fracture_dir} — run 01_dataset_setup.ipynb first.")
DATA_YAML = yaml_files[0]

print(f"Dataset YAML : {DATA_YAML}")
print(f"Runs root    : {RUNS_ROOT}")
print(f"Weights root : {WEIGHTS_ROOT}")

In [ ]:
# Install inside hosted notebook runtimes when needed.
# In local development, prefer installing dependencies from pyproject.toml instead.
if RUN_ENV in {"colab", "kaggle"}:
    %pip install -U ultralytics roboflow


In [ ]:
from ultralytics import YOLO
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Switch to a GPU runtime before serious training.")


In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {DATA_YAML}\nRun 01_dataset_setup.ipynb first.")

print(DATA_YAML.read_text())

In [ ]:
model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(RUNS_ROOT / "fracture"),
    name="train",
    exist_ok=True,
    seed=42,
)


In [ ]:
best_weight = RUNS_ROOT / "fracture" / "train" / "weights" / "best.pt"
final_weight = WEIGHTS_ROOT / "fracture_yolov8n_best.pt"
if not best_weight.exists():
    raise FileNotFoundError(f"Training did not produce best.pt at {best_weight}")
shutil.copy2(best_weight, final_weight)
print(f"Saved persistent weights to: {final_weight}")


## Next step

Open `03_evaluate_predict.ipynb` and set `WEIGHTS_PATH` to the saved path printed above.
